# Augmented LLM Agent — 场景演示

两个核心场景:

**场景1: CRAG 知识检索 + LLM 分析生成**
- 用户给出分析指令 → CRAG 从知识库检索相关内容
- 可选结合网页搜索补充 → LLM 生成分析报告

**场景2: CLI 操作任务**
- 用户给出操作需求 (创建目录、移动文件、搜索代码等)
- CLI Coder 安全执行命令

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import asyncio
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

from src.crag_retriever import CRAGRetriever
from src.chat_openai import ChatOpenAI
from src.coder_cli import CoderCLI
from src.utils import log_title, ensure_dir

ROOT = Path.cwd()
print(f"工作目录: {ROOT}")

---
## 场景1: CRAG 知识检索 + LLM 分析生成

用户输入分析需求 → CRAG 检索知识库 → LLM 生成分析报告

In [ ]:
# 初始化 CRAG 检索器并加载知识库
retriever = CRAGRetriever()

# 从 knowledge/ 目录加载
loaded = retriever.load_knowledge_dir('knowledge')
print(f"从 knowledge/ 加载了 {loaded} 个文档")

# 添加分析类文档
retriever.add_documents([
    "2024年AI行业报告: 大语言模型市场规模达$50B, 年增长率45%。"
    "主要玩家: OpenAI(GPT-4o), Anthropic(Claude 4), Google(Gemini)。"
    "企业应用以RAG和Agent为主要方向。",
    "2024年开发者调查: Python使用率67%居首, Rust满意度83%连续8年最受喜爱。"
    "AI/ML工具使用率达62%, 同比增长28%。",
    "数据工程实践: ETL模式(Apache Spark批处理, Kafka实时流, dbt数据转换)。"
    "数据治理需关注质量、安全、元数据管理和血缘追踪。",
    "软件架构演进: 单体→微服务→Serverless→AI原生。"
    "Agent驱动架构(ADA)兴起, RAG是连接LLM与企业数据的标准方案。",
])
print(f"总计 {retriever.doc_count} 个文档已索引")

In [ ]:
# 用户分析需求 (可自行修改)
USER_QUERY = (
    "请分析当前AI行业的发展趋势和开发者生态，"
    "结合数据工程和软件架构演进，"
    "给出2025年的技术展望和建议。"
)
print(f"[用户指令]\n{USER_QUERY}")

In [ ]:
# CRAG 检索
print("\n[CRAG 检索] 正在检索...")
result = retriever.retrieve(USER_QUERY, top_k=5, enable_correction=True)

print(f"检索结果: {len(result.docs)} 个文档片段")
for doc in result.docs:
    print(f"  [{doc.relevance.value}] score={doc.score:.2f} | {doc.content[:80]}...")

print(f"\n操作日志:")
for log in result.action_log:
    print(f"  → {log}")

context = result.corrected_context
print(f"\n上下文长度: {len(context)} 字符")

In [ ]:
# LLM 分析生成
print("\n[LLM 分析] 基于 CRAG 上下文生成...")

async def run_analysis():
    llm = ChatOpenAI(
        system_prompt=(
            "你是一个专业的数据分析师和技术顾问。"
            "请基于提供的上下文信息，给出结构化的分析和建议。"
            "如果上下文信息不足，请明确说明并基于你的知识补充。"
            "输出格式: 使用 Markdown，包含标题、要点和总结。"
        ),
        context=context[:4000],
        temperature=0.7,
    )
    response = await llm.chat(USER_QUERY)
    return response.content

try:
    analysis = await run_analysis()
    print(f"\n{'='*50}")
    print(analysis)
    print(f"{'='*50}")
    
    # 保存报告
    ensure_dir(ROOT / 'output')
    (ROOT / 'output' / 'crag_analysis_result.md').write_text(
        f"# CRAG 分析报告\n\n## 查询\n{USER_QUERY}\n\n"
        f"## 检索上下文\n{context[:3000]}\n\n"
        f"## 分析结果\n{analysis}",
        encoding='utf-8'
    )
    print("\n报告已保存到 output/crag_analysis_result.md")
except Exception as e:
    print(f"LLM 调用失败: {e}")
    print(f"\n[回退] CRAG 检索到的上下文:")
    print(context[:1000])

---
## 场景2: CLI 操作任务

用户给出操作需求 → CLI Coder 安全执行命令

演示: 创建目录、写文件、列表查看、搜索代码、统计信息

In [ ]:
coder = CoderCLI(str(ROOT))
output_dir = ensure_dir(ROOT / 'output' / 'cli_demo')

# 工具列表
print("可用 CLI 工具:")
for t in coder.get_tools():
    print(f"  - {t['name']}: {t['description']}")

In [ ]:
# 任务1: 创建项目目录结构
print("\n[任务1] 创建目录结构")
r = await coder.execute("run_command", command=f"mkdir -p {output_dir}/data/raw {output_dir}/data/processed {output_dir}/reports")
print(f"  结果: {r}")

In [ ]:
# 任务2: 创建示例数据文件
print("\n[任务2] 创建 sample.csv")
r = await coder.execute("write_file", 
    path=str(output_dir / "data" / "raw" / "sample.csv"),
    content="id,name,score,category\n1,Alice,95,AI\n2,Bob,87,Web\n3,Carol,92,AI\n"
)
print(f"  结果: {r}")

In [ ]:
# 任务3: 查看目录结构
print("\n[任务3] 查看目录")
r = await coder.execute("list_dir", path=str(output_dir), pattern="*")
print(r)

In [ ]:
# 任务4: 搜索代码中的 'CRAG'
print("\n[任务4] 搜索 CRAG 相关代码")
r = await coder.execute("search_code", pattern="CRAG", file_pattern="*.py")
print(r[:500])

In [ ]:
# 任务5: 读取数据文件
print("\n[任务5] 读取 sample.csv")
r = await coder.execute("read_file", path=str(output_dir / "data" / "raw" / "sample.csv"))
print(r)

In [ ]:
# 任务6: 统计项目代码行数
print("\n[任务6] 统计代码行数")
r = await coder.execute("run_command", 
    command="find . -name '*.py' -not -path './.git/*' -exec cat {} + | wc -l"
)
print(f"  总代码行数: {r.strip()}")

In [ ]:
# 任务7: 查看 Python 文件列表
print("\n[任务7] 列出项目中的 Python 文件")
r = await coder.execute("run_command",
    command="find . -name '*.py' -not -path './.git/*' | sort"
)
print(r)

---
## 总结

两个场景覆盖了 Agent 的核心能力:

| 场景 | 流程 | 核心组件 |
|------|------|----------|
| CRAG 分析 | 指令 → CRAG 检索 → LLM 生成 → 保存报告 | CRAGRetriever + ChatOpenAI |
| CLI 操作 | 指令 → CLI Coder 执行 → 查看结果 | CoderCLI (run_command/write_file/search_code...) |

完整 Agent 流程 (含 MCP): `python -m src.index`